In [ ]:
import time
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from PIL import ImageColor
from PIL import ImageDraw
from PIL import ImageFont
from PIL import ImageOps
import matplotlib.pyplot as plt
import sys
from collections import Counter


In [ ]:
# utils
def take_photo(camera):
    rval, frame = camera.read()
    return frame

def format_image(image):
    image_copy = image.copy()
    # image = image.reshape(1,640,640,3)
    image_copy = cv2.resize(image_copy,(640,640))
    image_copy = np.array(image_copy)
    image_copy = image_copy.reshape(1,640,640,3)
    return image_copy.copy()

def draw_bounding_box_on_image(
    image, ymin, xmin, ymax, xmax, color, font, thickness=4, display_str_list=()
):
    """Adds a bounding box to an image."""
    draw = ImageDraw.Draw(image)
    im_width, im_height = image.size
    (left, right, top, bottom) = (
        xmin * im_width,
        xmax * im_width,
        ymin * im_height,
        ymax * im_height,
    )
    draw.line(
        [(left, top), (left, bottom), (right, bottom), (right, top), (left, top)],
        width=thickness,
        fill=color,
    )

    # If the total height of the display strings added to the top of the bounding
    # box exceeds the top of the image, stack the strings below the bounding box
    # instead of above.
    display_str_heights = [font.getbbox(ds)[3] for ds in display_str_list]
    # Each display_str has a top and bottom margin of 0.05x.
    total_display_str_height = (1 + 2 * 0.05) * sum(display_str_heights)

    if top > total_display_str_height:
        text_bottom = top
    else:
        text_bottom = top + total_display_str_height
    # Reverse list and print from bottom to top.
    for display_str in display_str_list[::-1]:
        bbox = font.getbbox(display_str)
        text_width, text_height = bbox[2], bbox[3]
        margin = np.ceil(0.05 * text_height)
        draw.rectangle(
            [
                (left, text_bottom - text_height - 2 * margin),
                (left + text_width, text_bottom),
            ],
            fill=color,
        )
        draw.text(
            (left + margin, text_bottom - text_height - margin),
            display_str,
            fill="black",
            font=font,
        )
        text_bottom -= text_height - 2 * margin

def draw_boxes(image, boxes, class_names, scores, max_boxes=10, min_score=0.1):
    """Overlay labeled boxes on an image with formatted scores and label names."""
    colors = list(ImageColor.colormap.values())

    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Regular.ttf", 25
        )
    except IOError:
        print("Font not found, using default font.")
        font = ImageFont.load_default()

    for i in range(min(boxes.shape[0], max_boxes)):
        if scores[i] >= min_score and class_names[i] == 1.0:
            ymin, xmin, ymax, xmax = tuple(boxes[i])
            display_str = "{}: {}%".format(
                # class_names[i].decode("ascii"), int(100 * scores[i])
                class_names[i], int(100 * scores[i])
            )
            color = colors[hash(class_names[i]) % len(colors)]
            image_pil = Image.fromarray(np.uint8(image)).convert("RGB")
            draw_bounding_box_on_image(
                image_pil,
                ymin,
                xmin,
                ymax,
                xmax,
                color,
                font,
                display_str_list=[display_str],
            )
            np.copyto(image, np.array(image_pil))
    return image

In [ ]:


camera0 = cv2.VideoCapture('./test_video4.MOV')


# camera0 = cv2.VideoCapture(0)


video=cv2.VideoWriter('./video.avi',cv2.VideoWriter_fourcc(*'MJPG'),4,(640,640))

detector = tf.saved_model.load("../ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/saved_model")
# detector = tf.saved_model.load("faster-rcnn-openimages-v4-inception-resnet-v2")

def reduce_to_16_colors(rgb):
    # Define the 16 basic colors
    palette = [
        (0, 0, 0),          # Black
        (128, 0, 0),        # Dark Red
        (0, 128, 0),        # Dark Green
        (128, 128, 0),      # Dark Yellow
        (0, 0, 128),        # Dark Blue
        (128, 0, 128),      # Dark Magenta
        (0, 128, 128),      # Dark Cyan
        (192, 192, 192),    # Gray
        (128, 128, 128),    # Dark Gray
        (255, 0, 0),        # Red
        (0, 255, 0),        # Green
        (255, 255, 0),      # Yellow
        (0, 0, 255),        # Blue
        (255, 0, 255),      # Magenta
        (0, 255, 255),      # Cyan
        (255, 255, 255),    # White
    ]

    # Function to calculate Euclidean distance between two colors
    def color_distance(c1, c2):
        return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2 + (c1[2] - c2[2]) ** 2) ** 0.5

    # Find the closest color in the palette
    closest_color = min(palette, key=lambda c: color_distance(rgb, c))
    return closest_color


print("Start")

# frame[person[box_dimensions,tshirt_color]]

class Person:
    ymax: float
    ymin: float
    xmax: float
    xmin: float
    tshirt_color: tuple
    currently_visible: bool
    last_seen: time.time()

    def occurs_in_frames(self):
        frames = []
        for frame in range(len(detected_persons)):
            if self in detected_persons[frame]:
                frames.append(frame)
        return frames
    
detected_persons = []
current_persons_visible = []


c = 0
while True:
    img = take_photo(camera0)
    img = format_image(img)

    result = detector(img)
    result = {key:value.numpy() for key,value in result.items()}

    person_count = 0
    detected_persons.append([])
    for detected_object in range(result['detection_classes'].shape[1]):
        if result["detection_classes"][0][detected_object] == 1 and result["detection_scores"][0][detected_object] > 0.2:
            person_count += 1
            ymin, xmin, ymax, xmax = tuple(result["detection_boxes"][0][detected_object])
            detected_person_frame = img.reshape(640,640,3)[int(ymin*640):int(ymax*640), int(xmin*640):int(xmax*640)].copy()

            detected_person_cutout = detected_person_frame[int(0.3 * detected_person_frame.shape[1]):int(0.5 * detected_person_frame.shape[1]), int(0.4 * detected_person_frame.shape[0]):int(0.6 * detected_person_frame.shape[0])]

            if detected_person_cutout.shape[0] == 0 or detected_person_cutout.shape[1] == 0:
                detected_person_cutout = detected_person_frame

            pixel_colors = []
            for pixel_y, pixel_x in np.ndindex((detected_person_cutout.shape[0], detected_person_cutout.shape[1])):
                pb, pg, pr = detected_person_cutout[pixel_y][pixel_x]

                pb, pg, pr = reduce_to_16_colors((pb, pg, pr))
                pixel_colors.append((pb, pg, pr))

                detected_person_cutout[pixel_y][pixel_x] = [pb, pg, pr]
            most_common_color = Counter(pixel_colors).most_common(1)[0][0]
            detected_persons[c].append([[ymin,xmin,ymax,xmax],most_common_color])


            # print("Reduce color amount and count")
                
            
            # cv2.imwrite('results/frame_' + str(c) + '_person_' + str(person_count) + '.jpg', detected_person_frame)
            # cv2.imwrite('results/frame_' + str(c) + '_color_of_person_' + str(person_count) + '.jpg', np.array([[most_common_color]]))


    
    image_with_boxes = draw_boxes(
      img.reshape(640,640,3), result["detection_boxes"][0],
      result["detection_classes"][0], result["detection_scores"][0], min_score=0.2)

    video.write(image_with_boxes)

    if c > 5:
        break
    else:
        c += 1
    # image_with_boxes = draw_boxes(
    #   test.reshape(640,640,3), result["detection_boxes"][0],
    #   result["detection_classes"][0], result["detection_scores"][0])

print("End")

for frame_count in range(len(detected_persons)):
    print(frame_count)
    for person in detected_persons[frame_count]:
        print(person)

camera0.release()
video.release()
cv2.destroyAllWindows()